# CS5224 Data Curation — Stage 2E FINAL

## Final supermarket-fulfilment verification

This is the clean final Stage 2E.

Stage 2E does **not** perform further RedMart/Lazada browser scraping. During testing, Lazada's automated-access protection returned an anti-bot/CAPTCHA challenge instead of the normal product page. Therefore RedMart direct supermarket fulfilment cannot be verified automatically by the current pipeline.

The final policy is:

- **FairPrice** — reuse successful live verification from the earlier Stage 2E evidence.
- **Cold Storage** — reuse successful live verification from the earlier Stage 2E evidence.
- **RedMart** — retain `VERIFIED_DIRECT` only if explicit seller evidence identifying RedMart already exists; retain `NOT_DIRECT` only if explicit non-RedMart seller evidence already exists; otherwise classify as `UNKNOWN`.
- Unresolved RedMart rows receive the limitation code:
  `RETAILER_ANTI_BOT_PREVENTS_AUTOMATED_VERIFICATION`.

No CAPTCHA bypass, anti-bot bypass, or inferred RedMart fulfilment is performed.

The 1,300 canonical products remain valid product identities. Stage 2E only determines which retailer listings may be counted as strictly verified supermarket offers.


## 1. Imports and paths

In [1]:
from pathlib import Path
import hashlib
import re

import numpy as np
import pandas as pd


PROJECT_ROOT = Path.cwd().parent
CURATED_DATA_DIR = PROJECT_ROOT / "data" / "curated"

STAGE2D_SHORTLIST_FILE = (
    CURATED_DATA_DIR
    / "stage2d_provisional_seed_shortlist.csv"
)

STAGE2D_FULFILMENT_FILE = (
    CURATED_DATA_DIR
    / "stage2d_shortlist_fulfilment_review_working.csv"
)

# Reuse the live verification evidence generated by the earlier Stage 2E run.
AUTO_EVIDENCE_FILE = (
    CURATED_DATA_DIR
    / "stage2e_auto_fulfilment_evidence.csv"
)

# Final outputs.
FINAL_LISTINGS_FILE = (
    CURATED_DATA_DIR
    / "stage2e_final_listing_fulfilment.csv"
)

PRODUCT_COVERAGE_FILE = (
    CURATED_DATA_DIR
    / "stage2e_canonical_product_fulfilment_coverage.csv"
)

VERIFIED_DIRECT_OFFERS_FILE = (
    CURATED_DATA_DIR
    / "stage2e_verified_direct_offers.csv"
)

MANUAL_OVERRIDE_FILE = (
    CURATED_DATA_DIR
    / "stage2e_manual_fulfilment_override_working.csv"
)

QA_SAMPLE_FILE = (
    CURATED_DATA_DIR
    / "stage2e_final_fulfilment_qa_sample_working.csv"
)

RETAILER_SUMMARY_FILE = (
    CURATED_DATA_DIR
    / "stage2e_retailer_fulfilment_summary.csv"
)

CATEGORY_COVERAGE_FILE = (
    CURATED_DATA_DIR
    / "stage2e_category_fulfilment_coverage.csv"
)

REDMART_LIMITATION_FILE = (
    CURATED_DATA_DIR
    / "stage2e_redmart_verification_limitation.csv"
)

SUMMARY_FILE = (
    CURATED_DATA_DIR
    / "stage2e_final_summary.csv"
)


# Final project-level treatment of RedMart automated fulfilment verification.
REDMART_AUTOMATED_VERIFICATION_STATUS = (
    "RETAILER_ANTI_BOT_PREVENTS_AUTOMATED_VERIFICATION"
)

REDMART_AUTOMATED_VERIFICATION_NOTE = (
    "RedMart listings are hosted on Lazada. Automated live verification "
    "could not access the normal product page because Lazada returned an "
    "anti-bot/CAPTCHA challenge. Therefore unresolved RedMart fulfilment "
    "is retained as UNKNOWN and is not counted as a strict eligible offer."
)


required_files = [
    STAGE2D_SHORTLIST_FILE,
    STAGE2D_FULFILMENT_FILE,
    AUTO_EVIDENCE_FILE,
]

for file_path in required_files:
    print(
        f"{file_path.name:60s} -> "
        f"{file_path.exists()}"
    )
    assert file_path.exists(), (
        f"Required input missing: {file_path}"
    )

print(
    "\nPASS: Stage 2D inputs and cached Stage 2E live evidence found."
)


stage2d_provisional_seed_shortlist.csv                       -> True
stage2d_shortlist_fulfilment_review_working.csv              -> True
stage2e_auto_fulfilment_evidence.csv                         -> True

PASS: Stage 2D inputs and cached Stage 2E live evidence found.


## 2. Load Stage 2D shortlist, retailer listings, and cached live evidence

In [2]:
shortlist = pd.read_csv(
    STAGE2D_SHORTLIST_FILE,
    low_memory=False,
)

listing_queue = pd.read_csv(
    STAGE2D_FULFILMENT_FILE,
    dtype={
        "source_record_id": "string",
        "canonical_product_id": "string",
    },
    low_memory=False,
)

auto_evidence = pd.read_csv(
    AUTO_EVIDENCE_FILE,
    dtype={
        "source_record_id": "string",
        "canonical_product_id": "string",
    },
    low_memory=False,
)

auto_evidence = (
    auto_evidence
    .drop_duplicates(
        subset=["source_record_id"],
        keep="last",
    )
    .reset_index(drop=True)
)


required_listing_columns = [
    "source_record_id",
    "canonical_product_id",
    "retailer",
    "target_category",
    "product_name_raw",
    "stock_status",
    "offer_eligible",
    "product_url_raw",
]

missing_columns = [
    column
    for column in required_listing_columns
    if column not in listing_queue.columns
]

assert not missing_columns, (
    f"Missing listing columns: {missing_columns}"
)

assert (
    listing_queue["source_record_id"]
    .duplicated()
    .sum()
    == 0
), (
    "Expected one row per source listing."
)

print(
    "Selected canonical products:",
    f"{shortlist['canonical_product_id'].nunique():,}"
)

print(
    "Retailer listing rows:",
    f"{len(listing_queue):,}"
)

print(
    "Cached Stage 2E evidence rows:",
    f"{len(auto_evidence):,}"
)

print("\nListing rows by retailer:")
print(
    listing_queue[
        "retailer"
    ]
    .value_counts()
)


Selected canonical products: 1,300
Retailer listing rows: 2,650
Cached Stage 2E evidence rows: 2,650

Listing rows by retailer:
retailer
redmart        1132
fairprice      1131
coldstorage     387
Name: count, dtype: int64


## 3. Join cached live evidence to the selected retailer listing queue

In [3]:
evidence_columns = [
    "source_record_id",
    "http_status",
    "final_url",
    "page_title",
    "page_identity_score",
    "fetch_error",
    "block_or_challenge_detected",
    "seller_name",
    "seller_evidence",
    "auto_fulfilment_status",
    "auto_confidence",
    "verification_method",
    "auto_evidence",
    "verification_run_timestamp_utc",
]

available_evidence_columns = [
    column
    for column in evidence_columns
    if column in auto_evidence.columns
]

listings = (
    listing_queue
    .merge(
        auto_evidence[
            available_evidence_columns
        ],
        on="source_record_id",
        how="left",
    )
)

listings[
    "auto_fulfilment_status"
] = (
    listings[
        "auto_fulfilment_status"
    ]
    .fillna("UNKNOWN")
    .astype("string")
)

print(
    "Joined listing rows:",
    f"{len(listings):,}"
)

print(
    "\nCached automatic status by retailer:"
)

display(
    pd.crosstab(
        listings["retailer"],
        listings["auto_fulfilment_status"],
        margins=True,
    )
)


Joined listing rows: 2,650

Cached automatic status by retailer:


auto_fulfilment_status,UNKNOWN,VERIFIED_DIRECT,All
retailer,,,
coldstorage,0,387,387
fairprice,1,1130,1131
redmart,1132,0,1132
All,1133,1517,2650


## 4. Apply the final fulfilment policy

### FairPrice / Cold Storage

Existing successful live verification is reused.

### RedMart

RedMart is handled conservatively:

```text
explicit seller = RedMart
    -> VERIFIED_DIRECT

explicit seller = another seller
    -> NOT_DIRECT

otherwise
    -> UNKNOWN
    -> RETAILER_ANTI_BOT_PREVENTS_AUTOMATED_VERIFICATION
```

A Lazada URL or a RedMart source label alone is not treated as proof of direct supermarket fulfilment.


In [4]:
def norm(value):
    if pd.isna(value):
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(value)
        .strip()
        .casefold(),
    )


def redmart_explicit_direct(row):
    seller = norm(
        row.get(
            "seller_name",
            pd.NA,
        )
    )

    seller_evidence = norm(
        row.get(
            "seller_evidence",
            pd.NA,
        )
    )

    method = norm(
        row.get(
            "verification_method",
            pd.NA,
        )
    )

    seller_is_redmart = (
        seller
        in {
            "redmart",
            "red mart",
        }
    )

    has_explicit_marker = (
        "explicit" in seller_evidence
        or
        "explicit" in method
    )

    return (
        seller_is_redmart
        and has_explicit_marker
    )


def redmart_explicit_non_direct(row):
    seller_evidence = norm(
        row.get(
            "seller_evidence",
            pd.NA,
        )
    ).replace(
        " ",
        "_",
    )

    method = norm(
        row.get(
            "verification_method",
            pd.NA,
        )
    ).replace(
        " ",
        "_",
    )

    return (
        "explicit_non_redmart_seller"
        in seller_evidence
        or
        "explicit_non_redmart_seller"
        in method
    )


def apply_final_policy(row):
    retailer = norm(
        row[
            "retailer"
        ]
    )

    auto_status = norm(
        row.get(
            "auto_fulfilment_status",
            "unknown",
        )
    ).upper()

    # ----------------------------------------------------------
    # REDMART
    # ----------------------------------------------------------
    if retailer == "redmart":

        if redmart_explicit_direct(
            row
        ):
            return pd.Series(
                {
                    "policy_fulfilment_status":
                        "VERIFIED_DIRECT",
                    "verification_access_status":
                        "LIVE_EXPLICIT_SELLER_EVIDENCE",
                    "final_verification_method":
                        "EXPLICIT_REDMART_SELLER_EVIDENCE",
                    "verification_limitation":
                        pd.NA,
                }
            )

        if redmart_explicit_non_direct(
            row
        ):
            return pd.Series(
                {
                    "policy_fulfilment_status":
                        "NOT_DIRECT",
                    "verification_access_status":
                        "LIVE_EXPLICIT_SELLER_EVIDENCE",
                    "final_verification_method":
                        "EXPLICIT_NON_REDMART_SELLER_EVIDENCE",
                    "verification_limitation":
                        pd.NA,
                }
            )

        return pd.Series(
            {
                "policy_fulfilment_status":
                    "UNKNOWN",
                "verification_access_status":
                    REDMART_AUTOMATED_VERIFICATION_STATUS,
                "final_verification_method":
                    "REDMART_AUTOMATED_VERIFICATION_BLOCKED",
                "verification_limitation":
                    REDMART_AUTOMATED_VERIFICATION_NOTE,
            }
        )

    # ----------------------------------------------------------
    # FAIRPRICE / COLD STORAGE / OTHER SUPPORTED RETAILERS
    # ----------------------------------------------------------
    if auto_status == "VERIFIED_DIRECT":
        return pd.Series(
            {
                "policy_fulfilment_status":
                    "VERIFIED_DIRECT",
                "verification_access_status":
                    "LIVE_PAGE_VERIFIED",
                "final_verification_method":
                    row.get(
                        "verification_method",
                        "CACHED_STAGE2E_LIVE_EVIDENCE",
                    ),
                "verification_limitation":
                    pd.NA,
            }
        )

    if auto_status == "NOT_DIRECT":
        return pd.Series(
            {
                "policy_fulfilment_status":
                    "NOT_DIRECT",
                "verification_access_status":
                    "LIVE_PAGE_VERIFIED",
                "final_verification_method":
                    row.get(
                        "verification_method",
                        "CACHED_STAGE2E_LIVE_EVIDENCE",
                    ),
                "verification_limitation":
                    pd.NA,
            }
        )

    return pd.Series(
        {
            "policy_fulfilment_status":
                "UNKNOWN",
            "verification_access_status":
                "LIVE_EVIDENCE_INCONCLUSIVE",
            "final_verification_method":
                row.get(
                    "verification_method",
                    "CACHED_STAGE2E_INCONCLUSIVE",
                ),
            "verification_limitation":
                "Existing live evidence was inconclusive.",
        }
    )


policy_results = listings.apply(
    apply_final_policy,
    axis=1,
)

listings = pd.concat(
    [
        listings,
        policy_results,
    ],
    axis=1,
)

print(
    "Final policy status by retailer:"
)

display(
    pd.crosstab(
        listings["retailer"],
        listings["policy_fulfilment_status"],
        margins=True,
    )
)


Final policy status by retailer:


policy_fulfilment_status,UNKNOWN,VERIFIED_DIRECT,All
retailer,,,
coldstorage,0,387,387
fairprice,1,1130,1131
redmart,1132,0,1132
All,1133,1517,2650


## 5. Optional audited manual override

If stronger evidence becomes available later, for example from an authorised API/feed or audited manual review, it can override an `UNKNOWN` listing.

Allowed values:

```text
UNREVIEWED
VERIFIED_DIRECT
NOT_DIRECT
UNKNOWN
```

A manual override is applied only when both reviewer and evidence are filled.


In [5]:
manual_columns = [
    "manual_fulfilment_status",
    "manual_reviewer",
    "manual_evidence",
    "manual_notes",
]

manual_base = (
    listings.loc[
        listings[
            "policy_fulfilment_status"
        ]
        == "UNKNOWN"
    ]
    .copy()
)

for column in manual_columns:
    manual_base[
        column
    ] = pd.NA

manual_base[
    "manual_fulfilment_status"
] = "UNREVIEWED"


if MANUAL_OVERRIDE_FILE.exists():
    previous_manual = pd.read_csv(
        MANUAL_OVERRIDE_FILE,
        dtype={
            "source_record_id": "string",
        },
        low_memory=False,
    )

    previous_columns = [
        "source_record_id",
        *manual_columns,
    ]

    previous_columns = [
        column
        for column in previous_columns
        if column in previous_manual.columns
    ]

    previous_manual = (
        previous_manual[
            previous_columns
        ]
        .drop_duplicates(
            subset=["source_record_id"],
            keep="last",
        )
    )

    manual_review = (
        manual_base
        .drop(
            columns=manual_columns,
            errors="ignore",
        )
        .merge(
            previous_manual,
            on="source_record_id",
            how="left",
        )
    )

    for column in manual_columns:
        if column not in manual_review.columns:
            manual_review[
                column
            ] = pd.NA

    manual_review[
        "manual_fulfilment_status"
    ] = (
        manual_review[
            "manual_fulfilment_status"
        ]
        .fillna("UNREVIEWED")
    )

else:
    manual_review = manual_base.copy()


valid_manual_statuses = {
    "UNREVIEWED",
    "VERIFIED_DIRECT",
    "NOT_DIRECT",
    "UNKNOWN",
}

invalid_manual_statuses = sorted(
    set(
        manual_review[
            "manual_fulfilment_status"
        ]
        .dropna()
        .astype(str)
        .unique()
    )
    - valid_manual_statuses
)

assert not invalid_manual_statuses, (
    f"Invalid manual statuses: {invalid_manual_statuses}"
)

manual_review.to_csv(
    MANUAL_OVERRIDE_FILE,
    index=False,
    encoding="utf-8",
)

print(
    "Manual override queue:",
    f"{len(manual_review):,}"
)


Manual override queue: 1,133


## 6. Apply valid manual overrides and build strict offer flags

In [6]:
manual_lookup = (
    manual_review[
        [
            "source_record_id",
            *manual_columns,
        ]
    ]
    .drop_duplicates(
        subset=["source_record_id"],
        keep="last",
    )
)

final_listings = (
    listings
    .merge(
        manual_lookup,
        on="source_record_id",
        how="left",
    )
)

final_listings[
    "final_fulfilment_status"
] = (
    final_listings[
        "policy_fulfilment_status"
    ]
    .astype("string")
)

final_listings[
    "final_fulfilment_source"
] = "AUTOMATED_POLICY"


manual_status = (
    final_listings[
        "manual_fulfilment_status"
    ]
    .astype("string")
)

manual_complete = (
    manual_status
    .isin(
        [
            "VERIFIED_DIRECT",
            "NOT_DIRECT",
            "UNKNOWN",
        ]
    )
    &
    final_listings[
        "manual_reviewer"
    ]
    .notna()
    &
    final_listings[
        "manual_evidence"
    ]
    .notna()
)

final_listings.loc[
    manual_complete,
    "final_fulfilment_status",
] = final_listings.loc[
    manual_complete,
    "manual_fulfilment_status",
]

final_listings.loc[
    manual_complete,
    "final_fulfilment_source",
] = "MANUAL_REVIEWED_OVERRIDE"


def to_boolean(series):
    return (
        series
        .astype("string")
        .str.strip()
        .str.casefold()
        .map(
            {
                "true": True,
                "false": False,
                "yes": True,
                "no": False,
                "1": True,
                "0": False,
            }
        )
        .fillna(False)
        .astype(bool)
    )


final_listings[
    "verified_direct_listing"
] = (
    final_listings[
        "final_fulfilment_status"
    ]
    == "VERIFIED_DIRECT"
)

final_listings[
    "source_snapshot_offer_eligible"
] = to_boolean(
    final_listings[
        "offer_eligible"
    ]
)

final_listings[
    "verified_active_offer_at_source_snapshot"
] = (
    final_listings[
        "verified_direct_listing"
    ]
    &
    final_listings[
        "source_snapshot_offer_eligible"
    ]
)

final_listings[
    "eligible_for_cartoptix_strict_offer"
] = (
    final_listings[
        "verified_active_offer_at_source_snapshot"
    ]
)

print(
    "Final fulfilment status:"
)

print(
    final_listings[
        "final_fulfilment_status"
    ]
    .value_counts(
        dropna=False
    )
)

print(
    "\nVerified active offer rows:",
    f"{int(final_listings['verified_active_offer_at_source_snapshot'].sum()):,}"
)


Final fulfilment status:
final_fulfilment_status
VERIFIED_DIRECT    1517
UNKNOWN            1133
Name: count, dtype: Int64

Verified active offer rows: 1,489


## 7. Build one-row-per-canonical-product fulfilment coverage

In [7]:
product_rows = []

for (
    canonical_product_id,
    group,
) in final_listings.groupby(
    "canonical_product_id"
):

    direct_retailers = (
        group.loc[
            group[
                "verified_direct_listing"
            ],
            "retailer",
        ]
        .nunique()
    )

    active_retailers = (
        group.loc[
            group[
                "verified_active_offer_at_source_snapshot"
            ],
            "retailer",
        ]
        .nunique()
    )

    unknown_retailers = (
        group.loc[
            group[
                "final_fulfilment_status"
            ]
            == "UNKNOWN",
            "retailer",
        ]
        .nunique()
    )

    has_redmart_limitation = (
        (
            group[
                "retailer"
            ]
            .astype("string")
            .str.casefold()
            == "redmart"
        )
        &
        (
            group[
                "verification_access_status"
            ]
            == REDMART_AUTOMATED_VERIFICATION_STATUS
        )
    ).any()

    product_rows.append(
        {
            "canonical_product_id":
                canonical_product_id,
            "target_category":
                group[
                    "target_category"
                ].iloc[0],
            "total_retailer_count":
                int(
                    group[
                        "retailer"
                    ].nunique()
                ),
            "verified_direct_retailer_count":
                int(
                    direct_retailers
                ),
            "verified_active_offer_retailer_count":
                int(
                    active_retailers
                ),
            "unknown_fulfilment_retailer_count":
                int(
                    unknown_retailers
                ),
            "has_redmart_anti_bot_limitation":
                bool(
                    has_redmart_limitation
                ),
            "meets_2plus_verified_direct_retailers":
                bool(
                    direct_retailers
                    >= 2
                ),
            "meets_2plus_verified_active_offers":
                bool(
                    active_retailers
                    >= 2
                ),
        }
    )


product_coverage = pd.DataFrame(
    product_rows
)

assert (
    product_coverage[
        "canonical_product_id"
    ]
    .nunique()
    == shortlist[
        "canonical_product_id"
    ]
    .nunique()
)

total_products = len(
    product_coverage
)

products_2plus_direct = int(
    product_coverage[
        "meets_2plus_verified_direct_retailers"
    ].sum()
)

products_2plus_active = int(
    product_coverage[
        "meets_2plus_verified_active_offers"
    ].sum()
)

rate_2plus_direct = (
    products_2plus_direct
    / total_products
)

rate_2plus_active = (
    products_2plus_active
    / total_products
)

print(
    "Canonical products assessed:",
    f"{total_products:,}"
)

print(
    "Products with 2+ verified-direct retailers:",
    f"{products_2plus_direct:,}",
    f"({rate_2plus_direct:.2%})",
)

print(
    "Products with 2+ verified active offers:",
    f"{products_2plus_active:,}",
    f"({rate_2plus_active:.2%})",
)


Canonical products assessed: 1,300
Products with 2+ verified-direct retailers: 218 (16.77%)
Products with 2+ verified active offers: 207 (15.92%)


## 8. Retailer summary, category coverage, and independent QA sample

In [8]:
retailer_summary = (
    final_listings
    .groupby(
        [
            "retailer",
            "final_fulfilment_status",
        ]
    )
    .size()
    .rename("listing_count")
    .reset_index()
)

category_coverage = (
    product_coverage
    .groupby(
        "target_category"
    )
    .agg(
        canonical_products=(
            "canonical_product_id",
            "nunique",
        ),
        products_2plus_verified_direct=(
            "meets_2plus_verified_direct_retailers",
            "sum",
        ),
        products_2plus_verified_active=(
            "meets_2plus_verified_active_offers",
            "sum",
        ),
        products_with_redmart_anti_bot_limitation=(
            "has_redmart_anti_bot_limitation",
            "sum",
        ),
    )
    .reset_index()
)

category_coverage[
    "verified_direct_2plus_rate"
] = (
    category_coverage[
        "products_2plus_verified_direct"
    ]
    /
    category_coverage[
        "canonical_products"
    ]
)

category_coverage[
    "verified_active_2plus_rate"
] = (
    category_coverage[
        "products_2plus_verified_active"
    ]
    /
    category_coverage[
        "canonical_products"
    ]
)


def stable_hash(value):
    return int(
        hashlib.sha256(
            str(value)
            .encode("utf-8")
        )
        .hexdigest()[:16],
        16,
    )


qa_source = (
    final_listings.loc[
        (
            final_listings[
                "final_fulfilment_source"
            ]
            == "AUTOMATED_POLICY"
        )
        &
        (
            final_listings[
                "final_fulfilment_status"
            ]
            == "VERIFIED_DIRECT"
        )
    ]
    .copy()
)

qa_source[
    "_qa_hash"
] = qa_source[
    "source_record_id"
].map(
    stable_hash
)

qa_parts = []

for retailer, group in qa_source.groupby(
    "retailer"
):
    qa_parts.append(
        group
        .sort_values(
            "_qa_hash"
        )
        .head(20)
    )

if qa_parts:
    qa_sample = (
        pd.concat(
            qa_parts,
            ignore_index=True,
        )
        .drop(
            columns=["_qa_hash"],
            errors="ignore",
        )
    )
else:
    qa_sample = qa_source.head(0)

for column in [
    "qa_fulfilment_status",
    "qa_reviewer",
    "qa_evidence",
    "qa_notes",
]:
    if column not in qa_sample.columns:
        qa_sample[
            column
        ] = pd.NA

qa_sample[
    "qa_fulfilment_status"
] = (
    qa_sample[
        "qa_fulfilment_status"
    ]
    .fillna("UNREVIEWED")
)

display(
    retailer_summary
)

display(
    category_coverage
)

print(
    "Final QA sample rows:",
    f"{len(qa_sample):,}"
)


,retailer,final_fulfilment_status,listing_count
0,coldstorage,VERIFIED_DIRECT,387
1,fairprice,UNKNOWN,1
2,fairprice,VERIFIED_DIRECT,1130
3,redmart,UNKNOWN,1132


,target_category,canonical_products,products_2plus_verified_direct,products_2plus_verified_active,products_with_redmart_anti_bot_limitation,verified_direct_2plus_rate,verified_active_2plus_rate
0,Breakfast Foods,52,18,18,38,0.346154,0.346154
1,Canned & Packaged Food,174,40,40,144,0.229885,0.229885
2,Dairy,103,14,10,95,0.135922,0.097087
3,Drinks,154,19,18,137,0.123377,0.116883
4,Household Cleaning,104,4,2,103,0.038462,0.019231
5,Rice & Noodles,194,12,12,186,0.061856,0.061856
6,Snacks,431,59,56,393,0.136891,0.129930
7,Toiletries & Personal Care,88,52,51,36,0.590909,0.579545


Final QA sample rows: 40


## 9. Save RedMart limitation and final Stage 2E outputs

In [9]:
redmart_rows = (
    final_listings.loc[
        final_listings[
            "retailer"
        ]
        .astype("string")
        .str.casefold()
        == "redmart"
    ]
    .copy()
)

redmart_limitation = pd.DataFrame(
    [
        {
            "retailer":
                "redmart",
            "selected_listing_rows":
                len(
                    redmart_rows
                ),
            "verified_direct_rows":
                int(
                    (
                        redmart_rows[
                            "final_fulfilment_status"
                        ]
                        == "VERIFIED_DIRECT"
                    ).sum()
                ),
            "not_direct_rows":
                int(
                    (
                        redmart_rows[
                            "final_fulfilment_status"
                        ]
                        == "NOT_DIRECT"
                    ).sum()
                ),
            "unknown_rows":
                int(
                    (
                        redmart_rows[
                            "final_fulfilment_status"
                        ]
                        == "UNKNOWN"
                    ).sum()
                ),
            "limitation_code":
                REDMART_AUTOMATED_VERIFICATION_STATUS,
            "limitation_note":
                REDMART_AUTOMATED_VERIFICATION_NOTE,
            "treatment":
                (
                    "Keep unresolved RedMart fulfilment as UNKNOWN "
                    "and do not count it as a strict eligible offer."
                ),
            "future_resolution":
                (
                    "Use an authorised retailer/API/feed integration "
                    "or audited manual evidence if available later."
                ),
        }
    ]
)


verified_direct_offers = (
    final_listings.loc[
        final_listings[
            "verified_direct_listing"
        ]
    ]
    .copy()
)


final_listings.to_csv(
    FINAL_LISTINGS_FILE,
    index=False,
    encoding="utf-8",
)

product_coverage.to_csv(
    PRODUCT_COVERAGE_FILE,
    index=False,
    encoding="utf-8",
)

verified_direct_offers.to_csv(
    VERIFIED_DIRECT_OFFERS_FILE,
    index=False,
    encoding="utf-8",
)

retailer_summary.to_csv(
    RETAILER_SUMMARY_FILE,
    index=False,
    encoding="utf-8",
)

category_coverage.to_csv(
    CATEGORY_COVERAGE_FILE,
    index=False,
    encoding="utf-8",
)

qa_sample.to_csv(
    QA_SAMPLE_FILE,
    index=False,
    encoding="utf-8",
)

redmart_limitation.to_csv(
    REDMART_LIMITATION_FILE,
    index=False,
    encoding="utf-8",
)


summary = pd.DataFrame(
    [
        {
            "metric":
                "TOTAL_CANONICAL_PRODUCTS",
            "count":
                total_products,
            "value":
                pd.NA,
        },
        {
            "metric":
                "TOTAL_RETAILER_LISTINGS",
            "count":
                len(
                    final_listings
                ),
            "value":
                pd.NA,
        },
        {
            "metric":
                "VERIFIED_DIRECT_LISTINGS",
            "count":
                int(
                    final_listings[
                        "verified_direct_listing"
                    ].sum()
                ),
            "value":
                pd.NA,
        },
        {
            "metric":
                "UNKNOWN_FULFILMENT_LISTINGS",
            "count":
                int(
                    (
                        final_listings[
                            "final_fulfilment_status"
                        ]
                        == "UNKNOWN"
                    ).sum()
                ),
            "value":
                pd.NA,
        },
        {
            "metric":
                "PRODUCTS_2PLUS_VERIFIED_DIRECT",
            "count":
                products_2plus_direct,
            "value":
                pd.NA,
        },
        {
            "metric":
                "RATE_2PLUS_VERIFIED_DIRECT",
            "count":
                pd.NA,
            "value":
                rate_2plus_direct,
        },
        {
            "metric":
                "PRODUCTS_2PLUS_VERIFIED_ACTIVE",
            "count":
                products_2plus_active,
            "value":
                pd.NA,
        },
        {
            "metric":
                "RATE_2PLUS_VERIFIED_ACTIVE",
            "count":
                pd.NA,
            "value":
                rate_2plus_active,
        },
        {
            "metric":
                "STRICT_70_PERCENT_ACTIVE_COVERAGE_MET",
            "count":
                pd.NA,
            "value":
                bool(
                    rate_2plus_active
                    >= 0.70
                ),
        },
    ]
)

summary.to_csv(
    SUMMARY_FILE,
    index=False,
    encoding="utf-8",
)

display(
    redmart_limitation
)

print(
    "Saved all final Stage 2E outputs."
)


,retailer,selected_listing_rows,verified_direct_rows,not_direct_rows,unknown_rows,limitation_code,limitation_note,treatment,future_resolution
0,redmart,1132,0,0,1132,RETAILER_ANTI_BOT_PREVENTS_AUTOMATED_VERIFICATION,RedMart listings are hosted on Lazada. Automat...,Keep unresolved RedMart fulfilment as UNKNOWN ...,Use an authorised retailer/API/feed integratio...


Saved all final Stage 2E outputs.


## 10. Final validation

In [10]:
total_listings = len(
    final_listings
)

verified_direct_count = int(
    (
        final_listings[
            "final_fulfilment_status"
        ]
        == "VERIFIED_DIRECT"
    ).sum()
)

not_direct_count = int(
    (
        final_listings[
            "final_fulfilment_status"
        ]
        == "NOT_DIRECT"
    ).sum()
)

unknown_count = int(
    (
        final_listings[
            "final_fulfilment_status"
        ]
        == "UNKNOWN"
    ).sum()
)

manual_override_count = int(
    (
        final_listings[
            "final_fulfilment_source"
        ]
        == "MANUAL_REVIEWED_OVERRIDE"
    ).sum()
)

redmart_unknown_count = int(
    (
        (
            final_listings[
                "retailer"
            ]
            .astype("string")
            .str.casefold()
            == "redmart"
        )
        &
        (
            final_listings[
                "final_fulfilment_status"
            ]
            == "UNKNOWN"
        )
    ).sum()
)


print("=" * 112)
print("STAGE 2E FINAL VALIDATION — STRICT SUPERMARKET FULFILMENT")
print("=" * 112)

print(
    f"Selected canonical products                         : "
    f"{total_products:,}"
)

print(
    f"Retailer listing rows                               : "
    f"{total_listings:,}"
)

print("-" * 112)

print(
    f"VERIFIED_DIRECT listings                            : "
    f"{verified_direct_count:,}"
)

print(
    f"NOT_DIRECT listings                                 : "
    f"{not_direct_count:,}"
)

print(
    f"UNKNOWN fulfilment listings                         : "
    f"{unknown_count:,}"
)

print(
    f"Manual reviewed overrides                           : "
    f"{manual_override_count:,}"
)

print("-" * 112)

print(
    f"Products with 2+ verified-direct retailers          : "
    f"{products_2plus_direct:,}"
)

print(
    f"2+ verified-direct retailer rate                    : "
    f"{rate_2plus_direct:.2%}"
)

print(
    f"Products with 2+ verified ACTIVE offers             : "
    f"{products_2plus_active:,}"
)

print(
    f"2+ verified ACTIVE-offer rate                       : "
    f"{rate_2plus_active:.2%}"
)

print(
    f"Strict 70% active-offer target currently met        : "
    f"{rate_2plus_active >= 0.70}"
)

print("-" * 112)

print(
    f"RedMart unresolved fulfilment listings              : "
    f"{redmart_unknown_count:,}"
)

print(
    f"RedMart automated verification status               : "
    f"{REDMART_AUTOMATED_VERIFICATION_STATUS}"
)

print(
    "RedMart note                                      : "
    "Direct fulfilment cannot be automatically verified "
    "because Lazada anti-bot/CAPTCHA protection blocks "
    "access to the normal product page."
)

print("=" * 112)


assert (
    total_products
    == shortlist[
        "canonical_product_id"
    ].nunique()
)

assert (
    total_listings
    == len(
        listing_queue
    )
)

assert (
    verified_direct_count
    + not_direct_count
    + unknown_count
    == total_listings
)

allowed_statuses = {
    "VERIFIED_DIRECT",
    "NOT_DIRECT",
    "UNKNOWN",
}

assert set(
    final_listings[
        "final_fulfilment_status"
    ]
    .dropna()
    .unique()
).issubset(
    allowed_statuses
)

print()
print(
    "PASS: Final Stage 2E completed conservatively."
)

print(
    "RedMart unresolved listings remain UNKNOWN and are "
    "not counted as strict eligible retailer offers."
)


STAGE 2E FINAL VALIDATION — STRICT SUPERMARKET FULFILMENT
Selected canonical products                         : 1,300
Retailer listing rows                               : 2,650
----------------------------------------------------------------------------------------------------------------
VERIFIED_DIRECT listings                            : 1,517
NOT_DIRECT listings                                 : 0
UNKNOWN fulfilment listings                         : 1,133
Manual reviewed overrides                           : 0
----------------------------------------------------------------------------------------------------------------
Products with 2+ verified-direct retailers          : 218
2+ verified-direct retailer rate                    : 16.77%
Products with 2+ verified ACTIVE offers             : 207
2+ verified ACTIVE-offer rate                       : 15.92%
Strict 70% active-offer target currently met        : False
------------------------------------------------------------------

## Final Stage 2E output files

```text
stage2e_final_listing_fulfilment.csv
stage2e_canonical_product_fulfilment_coverage.csv
stage2e_verified_direct_offers.csv
stage2e_manual_fulfilment_override_working.csv
stage2e_final_fulfilment_qa_sample_working.csv
stage2e_retailer_fulfilment_summary.csv
stage2e_category_fulfilment_coverage.csv
stage2e_redmart_verification_limitation.csv
stage2e_final_summary.csv
```

For the later CartOptix database, canonical product identity and retailer-offer state remain separate. A future authorised RedMart API/feed can replace `UNKNOWN` fulfilment without rebuilding the canonical-product catalogue.
